# Classification - BERT
## Data Preparation

In [2]:
dataset = [
    # Positive (25)
    ("This movie is a masterpiece!", "Positive"),
    ("Terrific!", "Positive"),
    ("Absolutely loved every minute of it.", "Positive"),
    ("A stunning achievement in modern cinema.", "Positive"),
    ("The performances were heartfelt and unforgettable.", "Positive"),
    ("Brilliant writing and direction from start to finish.", "Positive"),
    ("I laughed, I cried, I want to see it again.", "Positive"),
    ("Easily the best film I've seen this year.", "Positive"),
    ("Visually breathtaking and emotionally rich.", "Positive"),
    ("A rare gem that lingers long after the credits roll.", "Positive"),
    ("Sharp, funny, and surprisingly moving.", "Positive"),
    ("The lead actor absolutely carries this film.", "Positive"),
    ("Tight pacing and a satisfying ending.", "Positive"),
    ("Genuinely one of the most original stories I've seen in years.", "Positive"),
    ("A triumph on every level.", "Positive"),
    ("Charming, clever, and full of heart.", "Positive"),
    ("The cinematography alone is worth the ticket.", "Positive"),
    ("It's flawed in places, but its ambition more than makes up for it.", "Positive"),
    ("I went in skeptical and walked out a fan.", "Positive"),
    ("A masterclass in tension and atmosphere.", "Positive"),
    ("Funny without trying too hard, and surprisingly smart.", "Positive"),
    ("The chemistry between the leads is electric.", "Positive"),
    ("Beautifully shot and impeccably acted.", "Positive"),
    ("Hands down a must-watch.", "Positive"),
    ("It nails the landing in a way most films don't even attempt.", "Positive"),

    # Negative (25)
    ("Not worth watching!", "Negative"),
    ("A complete waste of two hours.", "Negative"),
    ("Painfully slow and badly written.", "Negative"),
    ("I wanted my money back halfway through.", "Negative"),
    ("The plot makes absolutely no sense.", "Negative"),
    ("Wooden acting and lifeless dialogue.", "Negative"),
    ("One of the worst films I've ever sat through.", "Negative"),
    ("Boring, predictable, and way too long.", "Negative"),
    ("The trailer was better than the entire movie.", "Negative"),
    ("Generic, soulless, and forgettable.", "Negative"),
    ("Bad pacing ruins what could have been a decent idea.", "Negative"),
    ("Skip it. Seriously.", "Negative"),
    ("The script feels like a first draft nobody bothered to revise.", "Negative"),
    ("Visually flat and emotionally empty.", "Negative"),
    ("I checked my phone more than I watched the screen.", "Negative"),
    ("A confused mess of half-baked ideas.", "Negative"),
    ("Even the soundtrack couldn't save this one.", "Negative"),
    ("The ending is so bad it retroactively ruins the rest.", "Negative"),
    ("Oh sure, another two hours I'll never get back.", "Negative"),
    ("Charmless, joyless, and shockingly dull.", "Negative"),
    ("I have never been so bored in a theater.", "Negative"),
    ("Cliché after cliché, with nothing new to offer.", "Negative"),
    ("It tries so hard to be clever and fails every single time.", "Negative"),
    ("Don't believe the hype — this one's a dud.", "Negative"),
    ("Forgettable the moment the credits roll.", "Negative"),
]

In [4]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/Users/joewilkinson/Projects/whyyy/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
max_length = 128
formatted_data = [(f"[CLS] {text} [SEP]", label) for text, label in dataset]
tokenized_data = tokenizer(formatted_data, padding=True, truncation=True, max_length=max_length, return_tensors='pt')

In [8]:
import torch
from sklearn.preprocessing import LabelEncoder

input_ids = tokenized_data['input_ids']
attention_mask = tokenized_data['attention_mask']
labels = torch.tensor(LabelEncoder().fit_transform([label for _, label in dataset]))

In [9]:
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask = train_test_split(
    input_ids, labels, attention_mask, random_state=42, test_size=0.1
)

Create custom dataset and dataloaders.

In [10]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask':
                 self.attention_mask[idx],
                'labels': self.labels[idx]}

In [12]:
from torch.utils.data import DataLoader

batch_size = 4
train_dataset = CustomDataset(train_inputs, train_mask, train_labels)
train_dataloader = DataLoader(train_dataset,batch_size=batch_size, shuffle=True)

val_dataset = CustomDataset(val_inputs, val_mask, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [13]:
# Download model from Hugging Face
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13429.66it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo